In [0]:
from pyspark.sql.functions import col, sum

gold_df = spark.table("uci_retail.silver.online_retail")


In [0]:
vendas_por_pais = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .groupBy("Country")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_pais)

In [0]:
from pyspark.sql.functions import col, sum, round

vendas_por_pais = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .groupBy("Country")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_pais)

In [0]:
vendas_por_pais.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_pais")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_pais")
)

In [0]:
vendas_por_produto = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .groupBy("StockCode", "Description")
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_produto.limit(20))

In [0]:
display(
    vendas_por_produto
    .select(
        "StockCode",
        "Description",
        "TotalQuantity",
        "TotalRevenue"
    )
    .limit(20)
)

In [0]:
vendas_por_produto.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_item")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_item")
)

In [0]:
from pyspark.sql.functions import date_format

vendas_por_periodo = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .withColumn(
        "YearMonth",
        date_format(col("InvoiceDate"), "yyyy-MM")
    )
    .groupBy("YearMonth")
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy("YearMonth")
)

In [0]:
display(vendas_por_periodo)

In [0]:
vendas_por_periodo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_periodo")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_periodo")
)

In [0]:
from pyspark.sql.functions import countDistinct

vendas_por_cliente = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid") &
        col("CustomerID").isNotNull()
    )
    .groupBy("CustomerID")
    .agg(
        countDistinct("InvoiceNo").alias("TotalOrders"),
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_cliente.limit(20))

In [0]:
vendas_por_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_cliente")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_cliente")
)

In [0]:
# Pipeline de Engenharia de Dados - UCI Online Retail